In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [2]:
tmbd_movies = pd.read_csv(r"D:\Ultimate Programming\Data Bases\Project Dataset\Tmbd\tmdb_5000_movies.csv")
tmbd_credits = pd.read_csv(r"D:\Ultimate Programming\Data Bases\Project Dataset\Tmbd\tmdb_5000_credits.csv")

In [3]:
tmbd_movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
tmbd_credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [5]:
tmbd_movies = tmbd_movies.merge(tmbd_credits, on='title')

In [6]:
tmbd_movies.shape

(4809, 23)

In [7]:
df = tmbd_movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [8]:
df.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [9]:
df.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [10]:
df.dropna(inplace=True)

In [11]:
print(df.isnull().sum().sum())

0


In [12]:
df.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [13]:
import ast

def convert_data(obj):
    li = []
    for i in ast.literal_eval(obj):
        li.append(i['name'])
    return li

In [14]:
df['genres'] = df['genres'].apply(convert_data)

In [15]:
df['keywords'] = df['keywords'].apply(convert_data)

In [16]:
import ast

def convert_cast(obj):
    li = []
    count = 0
    for i in ast.literal_eval(obj):
        if count is not 3:
            li.append(i['name'])
            count += 1
        else: break
    return li

In [17]:
df['cast'] = df['cast'].apply(convert_cast)

In [18]:
def fetch_director(text):
    li = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            li.append(i['name'])
    return li 

In [19]:
df['crew'] = df['crew'].apply(fetch_director)

In [20]:
df.head(2)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


In [21]:
df['overview'] = df['overview'].apply(lambda x:x.split())

In [22]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [23]:
df['cast'] = df['cast'].apply(collapse)
df['crew'] = df['crew'].apply(collapse)
df['genres'] = df['genres'].apply(collapse)
df['keywords'] = df['keywords'].apply(collapse)

In [24]:
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']

In [25]:
df = df.drop(columns=['overview','genres','keywords','cast','crew'])

In [26]:
df.head(1)

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [27]:
df['tags'] = df['tags'].apply(lambda x: " ".join(x))

In [28]:
df['tags'] = df['tags'].apply(lambda x: x.lower())

In [29]:
df.head(1)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


In [30]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [31]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [32]:
df['tags'] = df['tags'].apply(stem)

In [33]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [34]:
scaler = cv.fit_transform(df['tags']).toarray()

In [35]:
scaler.shape

(4806, 5000)

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
similarity = cosine_similarity(scaler)

In [38]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ])

In [39]:
def recommand(movie):
    if movie not in df['title'].values:
        print("Movie not found in the dataset!")
        return
    index = df[df['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])), key=lambda x: x[1], reverse=True)
    for i in distances[1:6]:
        print(df.iloc[i[0]]['title'])

In [40]:
recommand('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [42]:
import joblib